<a href="https://colab.research.google.com/github/Basie12/medical_chatbot_ft_peft/blob/collabe_connect_branch/Medical_Chatbot_FT_PEFT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install trl
!pip install unsloth
!pip install bitsandbytes

In [2]:
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
max_seq_length =2048
load_in_4bit = True

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
# 4bit models from HF
fourbit_models = ["https://huggingface.co/unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
                  "https://huggingface.co/unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit",
                  "https://huggingface.co/unsloth/mistral-7b-instruct-v0.3-bnb-4bit"]
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B",
    max_seq_length = max_seq_length,
  dtype=None,
    load_in_4bit = load_in_4bit
)
medical_prompt = """Below is an instruction that describe a task, paired with an innput that provides further conteext.
Wriet response that appropriately completes the request.
Before answering, think carefully about the questionn and create a step-by-step chain of thoughts ensure
a logical and accurate response

### Instruction
You are a medical expert with advance knowledge inn clinical reasong, diagnosis, and treatment planning.
Answer the following medical question with clarity, accuracy, and patient safety in mind.
First, carefully analyze the problem step by step to ensure logical reasoning and then provide a clear and cloncise final response.
If relevant, highlight diagnostic considerations, treament opitions, and ref-flag safety notes.
Do not include irrevalnt details or speculative information. Please answer the following medical question.

### Question:
{}

### Chain of Thought:
<thinnk>
{}
>thinnk>

### Final response:
{}


"""

EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    inputs = examples["Question"]
    complex_cots = examples["Complex_CoT"]      # Chain-of-Thought reasoning
    outputs = examples["Response"]

    texts = []
    for question, cot, response in zip(inputs, complex_cots, outputs):
        # Add EOS token properly
        if not response.endswith(EOS_TOKEN):
            response = response + EOS_TOKEN

        text = medical_prompt.format(question, cot, response)
        texts.append(text)

    return {"text": texts}


==((====))==  Unsloth 2026.6.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/235 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.55k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

Unsloth: Will load unsloth/meta-llama-3.1-8b-unsloth-bnb-4bit as a legacy tokenizer.


In [4]:
# Load dataset
dataset = load_dataset(
    "FreedomIntelligence/medical-o1-reasoning-SFT",
    "en",
    split="train",   # or "test" depending on what you need
    trust_remote_code = True
)

# Apply formatting
dataset = dataset.map(formatting_prompts_func, batched=True)

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'FreedomIntelligence/medical-o1-reasoning-SFT' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'FreedomIntelligence/medical-o1-reasoning-SFT' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


README.md:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

medical_o1_sft.json:   0%|          | 0.00/58.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/19704 [00:00<?, ? examples/s]

Map:   0%|          | 0/19704 [00:00<?, ? examples/s]

In [5]:
print("Dataset size:", len(dataset))
print("Features:", dataset.features)
print("\n" + "="*80)

# Show first 3 examples
for i in range(3):
    print(f"\n🔹 EXAMPLE {i+1}")
    print("-" * 80)
    print("📌 QUESTION:\n", dataset[i]["Question"])
    print("\n🧠 COMPLEX_CoT (Reasoning):\n", dataset[i]["Complex_CoT"])
    print("\n💡 RESPONSE:\n", dataset[i]["Response"])
    print("="*100)

Dataset size: 19704
Features: {'Question': Value('string'), 'Complex_CoT': Value('string'), 'Response': Value('string'), 'text': Value('string')}


🔹 EXAMPLE 1
--------------------------------------------------------------------------------
📌 QUESTION:
 Given the symptoms of sudden weakness in the left arm and leg, recent long-distance travel, and the presence of swollen and tender right lower leg, what specific cardiac abnormality is most likely to be found upon further evaluation that could explain these findings?

🧠 COMPLEX_CoT (Reasoning):
 Okay, let's see what's going on here. We've got sudden weakness in the person's left arm and leg - and that screams something neuro-related, maybe a stroke?

But wait, there's more. The right lower leg is swollen and tender, which is like waving a big flag for deep vein thrombosis, especially after a long flight or sitting around a lot.

So, now I'm thinking, how could a clot in the leg end up causing issues like weakness or stroke symptoms?

Oh

In [6]:
## Pretraining Inference
sample_question = """ A 43-year-old man with diabetes mellitus presents with rapidly progressing skin changes on his leg and signs of systemic infection after a leg injury 4 days ago.
Given his vital signs indicating possible sepsis and impaired renal function, what is the most appropriate step to establish a definitive diagnosis of his condition?"""


In [7]:
correct_response  = """In a case involving a 43-year-old man with diabetes who is experiencing rapidly progressing
skin changes on his leg along with systemic infection signs after a recent injury, the clinical suspicion may
lean towards a serious condition such as necrotizing fasciitis. This is a rapidly advancing, life-threatening
soft tissue infection that requires immediate surgical intervention.Step-by-step reasoning (Complex_CoT style):

Patient has diabetes → higher risk for severe infections and poor wound healing.
Rapid progression over just 4 days + systemic signs (sepsis) → suggests deep tissue infection rather than simple cellulitis.
Impaired renal function + sepsis → indicates need for urgent diagnosis.
Definitive diagnosis of necrotizing fasciitis is primarily clinical + surgical exploration (not just imaging or labs).

Final Response:
The most appropriate step to establish a definitive diagnosis is emergent surgical exploration and debridement.
Tissue biopsy and culture during surgery can confirm the diagnosis, while delaying surgery can be life-threatening."""

In [8]:
medical_inference_prompt = """### Instruction:
You are a highly knowledgeable medical AI assistant. Provide accurate, step-by-step clinical reasoning
and a clear final answer.

### Question:
{}

### Response:"""

# For Alpaca-style / Unsloth
alpaca_style_prompt = """Below is an instruction that describes a task, paired with an input that provides
further context. Write a response that appropriately completes the request.

### Instruction:
Answer the following medical question with detailed step-by-step reasoning.

### Input:
{}

### Response:"""

In [9]:
from unsloth import FastLanguageModel
import torch

# Load your model (use the one you trained or base)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B",  # or your fine-tuned path
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

# Enable faster inference
FastLanguageModel.for_inference(model)

def generate_medical_response(question, max_new_tokens=512):
    prompt = medical_inference_prompt.format(question)

    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract only the generated part after "Response:"
    response = response.split("### Response:")[-1].strip()

    return response

# ========================
# Test with Sample Question
# ========================

question = """A 43-year-old man with diabetes mellitus presents with rapidly progressing skin changes on his leg and signs of systemic infection after a leg injury 4 days ago.
Given his vital signs indicating possible sepsis and impaired renal function, what is the most appropriate step to establish a definitive diagnosis of his condition?"""

print("🩺 Question:\n", question)
print("\n🤖 Model Response:\n")
print(generate_medical_response(question))

==((====))==  Unsloth 2026.6.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/meta-llama-3.1-8b-unsloth-bnb-4bit as a legacy tokenizer.


🩺 Question:
 A 43-year-old man with diabetes mellitus presents with rapidly progressing skin changes on his leg and signs of systemic infection after a leg injury 4 days ago.
Given his vital signs indicating possible sepsis and impaired renal function, what is the most appropriate step to establish a definitive diagnosis of his condition?

🤖 Model Response:



Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


In [10]:
print(generate_medical_response(sample_question))

Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


1. **Diagnose:** The patient has signs of sepsis and impaired renal function. He has a history of diabetes mellitus, which increases his risk of developing skin infections and other complications. The most appropriate step to establish a definitive diagnosis of his condition is to perform a comprehensive medical history and physical examination, including assessment of vital signs, review of medications, and evaluation of the affected limb for signs of infection and tissue damage. This will help to identify any underlying causes and guide appropriate management.

2. **Assessment:** Based on the provided information, the patient's vital signs indicate possible sepsis, which is characterized by an elevated heart rate, respiratory rate, and temperature, as well as low blood pressure. Additionally, the patient's renal function appears to be impaired, as indicated by elevated creatinine levels in his blood. These findings suggest that the patient may be suffering from a severe bacterial inf